In [1]:
import sys
import pandas as pd

sys.path.append("../src")   # must come BEFORE importing our own modules
import metrics
import sec_extract

financials = pd.read_csv("../data/processed/financials_long.csv")
adjustments = pd.read_csv("../data/manual/manual_adjustments.csv")
financials = metrics.apply_adjustments(financials, adjustments)
print("Manual adjustments applied:", len(adjustments))
wide = metrics.to_wide(financials)
results = metrics.compute_metrics(wide)
print("Company-years with metrics:", len(results))

Manual adjustments applied: 3
Company-years with metrics: 1640


In [2]:
mg = results[results["ticker"] == "MG"].set_index("fiscal_year")

# (fiscal year, metric, answer key value, allowed difference)
answer_key = [
    (2025, "ebitda", 71580, 1),             (2024, "ebitda", 72836, 1),             (2023, "ebitda", 32195, 1),
    (2025, "ebitda_margin", 0.0989, 0.0005), (2024, "ebitda_margin", 0.0998, 0.0005), (2023, "ebitda_margin", 0.0456, 0.0005),
    (2025, "fcf", 8307, 1),                 (2024, "fcf", 32227, 1),                (2023, "fcf", 5894, 1),
    (2025, "fcf_conversion", 0.1161, 0.0005),(2024, "fcf_conversion", 0.4425, 0.0005),(2023, "fcf_conversion", 0.1831, 0.0005),
    (2025, "capex_intensity", 0.0341, 0.0005),(2024, "capex_intensity", 0.0245, 0.0005),(2023, "capex_intensity", 0.0296, 0.0005),
    (2025, "total_debt", 202357, 1),        (2024, "total_debt", 190126, 1),
    (2025, "net_debt", 174349, 1),          (2024, "net_debt", 171809, 1),
    (2025, "net_leverage", 2.436, 0.005),   (2024, "net_leverage", 2.359, 0.005),
    (2025, "dso", 77.98, 0.05),             (2024, "dso", 63.67, 0.05),
    (2025, "nopat", 30429, 1),              (2024, "nopat", 29869.5, 1),            (2023, "nopat", -1428, 1),
    (2025, "invested_capital", 409453, 1),  (2024, "invested_capital", 370377, 1),
    (2025, "roic", 0.0780, 0.0005),
    (2025, "roic_ex_goodwill", 0.1472, 0.0005),
    (2025, "revenue_cagr_3y", 0.0175, 0.0005),
    (2025, "ebitda_margin_range_3y", 0.0542, 0.0005),
]

matches = 0
for year, metric, expected, tolerance in answer_key:
    actual = mg.loc[year, metric]
    ok = abs(actual - expected) <= tolerance
    matches += ok
    print(f"FY{year} {metric:24} key: {expected:>10}   python: {actual:>12.4f}   {'MATCH' if ok else 'MISMATCH'}")

print(f"\n{matches} of {len(answer_key)} match")

FY2025 ebitda                   key:      71580   python:   71580.0000   MATCH
FY2024 ebitda                   key:      72836   python:   72836.0000   MATCH
FY2023 ebitda                   key:      32195   python:   32195.0000   MATCH
FY2025 ebitda_margin            key:     0.0989   python:       0.0989   MATCH
FY2024 ebitda_margin            key:     0.0998   python:       0.0998   MATCH
FY2023 ebitda_margin            key:     0.0456   python:       0.0456   MATCH
FY2025 fcf                      key:       8307   python:    8307.0000   MATCH
FY2024 fcf                      key:      32227   python:   32227.0000   MATCH
FY2023 fcf                      key:       5894   python:    5894.0000   MATCH
FY2025 fcf_conversion           key:     0.1161   python:       0.1161   MATCH
FY2024 fcf_conversion           key:     0.4425   python:       0.4425   MATCH
FY2023 fcf_conversion           key:     0.1831   python:       0.1831   MATCH
FY2025 capex_intensity          key:     0.0341   py

In [3]:
results.to_csv("../data/processed/metrics.csv", index=False)

# The 8 companies that passed the size gate, in the year they were tested
gate = pd.read_csv("../data/processed/size_gate_results.csv")
shortlist = gate[gate["size_gate"] == "PASS"][["cik", "company_name", "fiscal_year"]]
latest = shortlist.merge(results, on=["cik", "fiscal_year"])

view = latest[["ticker", "company_name", "fiscal_year"]].copy()
view["revenue_$M"] = (latest["revenue"] / 1000).round(0)
view["ebitda_$M"] = (latest["ebitda"] / 1000).round(1)

percent_metrics = {"ebitda_margin": "margin_%", "revenue_cagr_3y": "cagr_3y_%",
                   "fcf_conversion": "fcf_conv_%", "capex_intensity": "capex_%",
                   "roic": "roic_%", "ebitda_margin_range_3y": "margin_swing_%",
                   "acq_spend_to_revenue": "acq_spend_%"}
for col, label in percent_metrics.items():
    view[label] = (latest[col] * 100).round(1)

view["leverage_x"] = latest["net_leverage"].round(2)
view["dso_days"] = latest["dso"].round(0)
view["debt_tags_missing"] = latest["debt_tags_missing"]

view.sort_values("ebitda_$M", ascending=False)

,ticker,company_name,fiscal_year,revenue_$M,ebitda_$M,margin_%,cagr_3y_%,fcf_conv_%,capex_%,roic_%,margin_swing_%,acq_spend_%,leverage_x,dso_days,debt_tags_missing
5,ADV,Advantage Solutions Inc.,2025.0,3543.0,75.8,2.1,-1.0,72.6,0.2,-4.5,9.1,0.0,18.91,61.0,False
3,MG,"Mistras Group, Inc.",2025.0,724.0,71.6,9.9,1.7,11.6,3.4,7.8,5.4,0.0,2.44,78.0,False
1,WLDN,"Willdan Group, Inc.",2025.0,682.0,62.8,9.2,16.7,112.5,1.4,12.2,1.7,6.6,-0.24,35.0,False
4,ONT,"Onterris, Inc.",2025.0,831.0,62.6,7.5,15.1,145.5,2.0,1.2,5.2,21.6,4.93,68.0,False
2,QNST,"QUINSTREET, INC",2026.0,1294.0,60.6,4.7,30.6,211.1,0.3,13.0,5.4,0.0,-0.96,51.0,False
6,RMR,RMR GROUP INC.,2025.0,183.0,53.3,29.2,-3.0,NaN,NaN,20.5,23.0,43.1,-0.33,NaN,False
7,TISI,TEAM INC,2025.0,896.0,48.2,5.4,2.2,-42.9,1.0,3.9,2.6,0.0,5.80,72.0,False
0,LQDT,LIQUIDITY SERVICES INC,2025.0,477.0,45.5,9.5,19.4,129.6,1.6,90.7,2.3,4.1,-3.84,9.0,True


In [4]:
inputs = ["revenue", "operating_income", "d_and_a", "cfo", "capex", "cash",
          "short_term_debt", "long_term_debt", "finance_lease_current",
          "finance_lease_noncurrent", "stockholders_equity", "accounts_receivable", "goodwill"]

problem = latest[latest["ticker"].isin(["TISI", "ADV", "LQDT", "RMR"])].set_index("ticker")
problem[inputs].T

ticker,LQDT,ADV,RMR,TISI
revenue,476669.0,3542642.0,182703.0,896483.0
operating_income,35066.0,-126466.0,41782.0,14071.0
d_and_a,10409.0,202258.0,11551.0,34082.0
cfo,66758.0,61532.0,75746.0,-11348.0
capex,7800.0,6477.0,NaN,9289.0
cash,174607.0,240850.0,62297.0,18145.0
short_term_debt,NaN,13250.0,NaN,NaN
long_term_debt,NaN,1660611.0,NaN,NaN
finance_lease_current,NaN,NaN,NaN,1689.0
finance_lease_noncurrent,NaN,NaN,NaN,6986.0


In [5]:
import json


def search_balance_sheet_tags(cik, fiscal_year, keywords):
    """List year-end (balance sheet) 10-K tags whose names contain any keyword, in $000s."""
    path = f"../data/raw/companyfacts/CIK{int(cik):010d}.json"
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    found = set()
    for tag, info in data["facts"].get("us-gaap", {}).items():
        if any(k in tag for k in keywords) and "USD" in info["units"]:
            for e in info["units"]["USD"]:
                if e.get("form") == "10-K" and "start" not in e:    # no start date = a snapshot
                    end = pd.Timestamp(e["end"])
                    if (end - pd.Timedelta(days=15)).year == fiscal_year:
                        found.add((tag, e["val"] / 1000))
    return sorted(found)


searches = {
    "TISI": ["Debt", "Borrowing", "LineOfCredit"],
    "LQDT": ["Cash"],
    "ADV": ["StockholdersEquity"],
    "RMR": ["Cash"],
}

for ticker, keywords in searches.items():
    row = latest[latest["ticker"] == ticker].iloc[0]
    print(f"\n{ticker} - {row['company_name']} (FY{int(row['fiscal_year'])})")
    for tag, value in search_balance_sheet_tags(row["cik"], int(row["fiscal_year"]), keywords):
        print(f"   {tag:80} {value:>14,.0f}")


TISI - TEAM INC (FY2025)
   DebtInstrumentCarryingAmount                                                            297,605
   LongTermDebt                                                                            288,526
   LongTermDebtAndCapitalLeaseObligations                                                  293,343
   LongTermDebtAndCapitalLeaseObligationsCurrent                                             3,858
   LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities                        297,201
   LongTermDebtMaturitiesRepaymentsOfPrincipalAfterYearFive                                      0
   LongTermDebtMaturitiesRepaymentsOfPrincipalInNextTwelveMonths                             2,169
   LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFive                                   227,958
   LongTermDebtMaturitiesRepaymentsOfPrincipalInYearFour                                     3,056
   LongTermDebtMaturitiesRepaymentsOfPrincipalInYearThree                          

In [6]:
# Answer key test: Mistras at the $20.35 deal price, using average diluted shares (as in Excel)
mg_latest = latest[latest["ticker"] == "MG"]
deal_price = pd.DataFrame({"ticker": ["MG"], "close_price": [20.35]})
mg_test = metrics.compute_valuation(mg_latest, deal_price)   # no shares passed -> diluted average

print("Equity value:     ", round(mg_test["market_cap"].iloc[0]), "  (answer key: 652,380)")
print("Enterprise value: ", round(mg_test["enterprise_value"].iloc[0]), "  (answer key: 826,729)")
print("EV / EBITDA:      ", round(mg_test["ev_to_ebitda"].iloc[0], 2), "  (answer key: 11.55)")

Equity value:      652380   (answer key: 652,380)
Enterprise value:  826729   (answer key: 826,729)
EV / EBITDA:       11.55   (answer key: 11.55)


In [7]:
import json

# Cover-page shares for each shortlisted company (from the cached SEC files)
share_rows = []
for cik in latest["cik"]:
    with open(f"../data/raw/companyfacts/CIK{int(cik):010d}.json", encoding="utf-8") as f:
        result = sec_extract.latest_shares_outstanding(json.load(f))
    if result:
        share_rows.append({"cik": cik, **result})
shares = pd.DataFrame(share_rows)

prices = pd.read_csv("../data/raw/share_prices.csv")
valuation = metrics.compute_valuation(latest, prices[["ticker", "close_price"]], shares)
valuation.to_csv("../data/processed/valuation.csv", index=False)

show = valuation[["ticker", "close_price", "shares_basis", "shares_as_of"]].copy()
show["shares_M"] = (valuation["shares_used_000s"] / 1000).round(1)
show["market_cap_$M"] = (valuation["market_cap"] / 1000).round(0)
show["net_debt_$M"] = (valuation["net_debt"] / 1000).round(0)
show["ev_$M"] = (valuation["enterprise_value"] / 1000).round(0)
show["ebitda_$M"] = (valuation["ebitda"] / 1000).round(1)
show["ev_ebitda_x"] = valuation["ev_to_ebitda"].round(2)
show.sort_values("ev_ebitda_x")

,ticker,close_price,shares_basis,shares_as_of,shares_M,market_cap_$M,net_debt_$M,ev_$M,ebitda_$M,ev_ebitda_x
6,RMR,17.48,avg diluted (fallback),NaN,16.6,291.0,-18.0,273.0,53.3,5.12
7,TISI,24.20,cover page,2026-08-06,4.6,111.0,279.0,390.0,48.2,8.10
3,MG,21.53,cover page,2026-08-05,31.8,686.0,174.0,860.0,71.6,12.02
4,ONT,14.83,cover page,2026-07-31,35.1,521.0,309.0,830.0,62.6,13.24
2,QNST,16.52,cover page,2026-08-21,57.1,943.0,-58.0,885.0,60.6,14.60
1,WLDN,72.48,cover page,2026-08-06,15.2,1104.0,-15.0,1089.0,62.8,17.33
5,ADV,30.28,cover page,2026-07-31,12.8,388.0,1433.0,1821.0,75.8,24.03
0,LQDT,44.85,cover page,2026-08-03,31.3,1405.0,-175.0,1231.0,45.5,27.06


In [8]:
prices

,ticker,price_date,close_price,price_source
0,ADV,2026-10-23,30.28,Nasdaq.com historical quotes
1,MG,2026-10-23,21.53,Nasdaq.com historical quotes
2,WLDN,2026-10-23,72.48,Nasdaq.com historical quotes
3,ONT,2026-10-23,14.83,Nasdaq.com historical quotes
4,QNST,2026-10-23,16.52,Nasdaq.com historical quotes
5,RMR,2026-10-23,17.48,Nasdaq.com historical quotes
6,TISI,2026-10-23,24.20,Nasdaq.com historical quotes
7,LQDT,2026-10-23,44.85,Nasdaq.com historical quotes
